# Tokenizer and BIO alignment
This notebook makes the word-to-subword mapping and Strategy A visible. Run from the repository root.

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer
from pathlib import Path
from lab9.ner_pipeline import align_labels

dataset = load_dataset('eriktks/conll2003')
label_names = dataset['train'].features['ner_tags'].feature.names
tokenizer = AutoTokenizer.from_pretrained('distilbert/distilbert-base-cased', use_fast=True)
row = dataset['train'][0]
encoded = tokenizer(row['tokens'], is_split_into_words=True, truncation=True)
word_ids = encoded.word_ids()
aligned = align_labels(word_ids, row['ner_tags'])
[(tokenizer.convert_ids_to_tokens(token_id), word_id, label_names[label] if label >= 0 else 'IGNORE')
 for token_id, word_id, label in zip(encoded['input_ids'], word_ids, aligned)]

## Inspect subword rates
The full run exports sentence length, entity status, piece count, and split indicator to `tokenization_analysis.csv`.

In [ ]:
import pandas as pd
stats = pd.read_csv(Path('lab9') / 'tokenization_analysis.csv')
stats.groupby(['sentence_length', 'entity'])['pieces'].agg(['count', 'mean']).round(3)